In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [1]:
# Helper functions
def add_user_message(messages, text) -> list[str]:
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text) -> list[str]:
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [4]:
# helper functions for running the test cases through claude

TASK_PROMPT = """\
Please solve the following task:

{task}
"""


def run_prompt(test_case) -> str:
    """Merges the prompt and test case input"""

    prompt = TASK_PROMPT.format(task=test_case["task"])
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


def run_test_case(test_case) -> float:
    """Calls run_prompt and grades the result"""
    output = run_prompt(test_case)

    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }


def run_eval(dataset):
    """Loads the dataset and runs run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [5]:
import json

with open("dataset.json", "r") as file:
    dataset = json.load(file)

results = run_eval(dataset)

In [6]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Python Function to Extract AWS Account ID from ARN\n\n```python\ndef extract_account_id_from_arn(arn: str) -> str:\n    \"\"\"\n    Extracts the AWS account ID from an ARN (Amazon Resource Name) string.\n    \n    ARN format: arn:partition:service:region:account-id:resource-type/resource-id\n    Example: arn:aws:s3:::my-bucket\n    Example: arn:aws:iam::123456789012:user/Development/product_1234/*\n    \n    Args:\n        arn (str): The ARN string to parse\n        \n    Returns:\n        str: The AWS account ID, or empty string if not found\n        \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        raise ValueError(\"ARN must be a non-empty string\")\n    \n    # Split the ARN by colons\n    parts = arn.split(\":\")\n    \n    # Valid ARN must have at least 6 parts\n    if len(parts) < 6:\n        raise ValueError(f\"Invalid ARN format: {arn}\")\n    \n    # The account ID is the 5t